<a href="https://colab.research.google.com/github/Nayaramiguell/Projetos-HTML/blob/main/notasPendentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np
import warnings

# Ocultar avisos chatos de atualização do Jupyter/Colab
warnings.filterwarnings("ignore", category=DeprecationWarning)

print("=== CONCILIAÇÃO FISCAL AUTOMATIZADA: NF-E + CTE ===")

# 1. LER OS TRÊS ARQUIVOS EXCEL
try:
    df_edocs = pd.read_excel("relatorio_edocs.xlsx")
    df_cte = pd.read_excel("relatorio_cte.xlsx")
    df_senior = pd.read_excel("relatorio_senior_x.xlsx")
    print("✅ Os três arquivos Excel foram carregados com sucesso!")
except Exception as e:
    print(f"❌ Erro crítico ao ler os arquivos: {e}")
    print("Verifique se os arquivos estão na pasta lateral com os nomes corretos.")

# LIMPEZA DE ESPAÇOS NOS CABEÇALHOS
df_edocs.columns = df_edocs.columns.str.strip()
df_cte.columns = df_cte.columns.str.strip()
df_senior.columns = df_senior.columns.str.strip()

# 2. PADRONIZAÇÃO E LIMPEZA DO RELATÓRIO DE NF-E (EDOCS)
df_edocs['Número da NF-e'] = pd.to_numeric(df_edocs['Número da NF-e'], errors='coerce')
df_edocs = df_edocs.dropna(subset=['Número da NF-e'])
df_edocs['Documento_Limpo'] = df_edocs['Número da NF-e'].astype(int)
df_edocs['Tipo_Doc'] = 'NF-e'

# 3. PADRONIZAÇÃO E LIMPEZA DO RELATÓRIO DE CTE (EDOCS)
coluna_numero_cte = 'Número da NF-e' if 'Número da NF-e' in df_cte.columns else df_cte.columns[0]
df_cte['Numero_Identificado'] = pd.to_numeric(df_cte[coluna_numero_cte], errors='coerce')
df_cte = df_cte.dropna(subset=['Numero_Identificado'])
df_cte['Documento_Limpo'] = df_cte['Numero_Identificado'].astype(int)
df_cte['Tipo_Doc'] = 'CT-e'

# 4. JUNÇÃO DAS DUAS FONTES DO EDOCS (NF-e + CT-e)
df_edocs_total = pd.concat([df_edocs, df_cte], ignore_index=True, join='outer')

# 5. FILTRO DA SITUAÇÃO (Código 6 = Autorizada no eDocs)
if 'Situação' in df_edocs_total.columns:
    df_edocs_total['Situação'] = pd.to_numeric(df_edocs_total['Situação'], errors='coerce')
    df_recebidos_autorizados = df_edocs_total[df_edocs_total['Situação'] == 6]
else:
    df_recebidos_autorizados = df_edocs_total

# 6. LIMPEZA DA BASE DO SENIOR X
df_senior['Nota Fiscal'] = pd.to_numeric(df_senior['Nota Fiscal'], errors='coerce')
df_senior = df_senior.dropna(subset=['Nota Fiscal'])
lista_notas_lancadas = df_senior['Nota Fiscal'].astype(int).tolist()

# 7. O CRUZAMENTO FINAL
df_pendentes = df_recebidos_autorizados[~df_recebidos_autorizados['Documento_Limpo'].isin(lista_notas_lancadas)]

print("\n================ RESULTADO DA AUDITORIA ================")
if not df_pendentes.empty:
    print(f"⚠️ Sucesso! Foram encontrados {len(df_pendentes)} documentos (NF-e/CT-e) SEM LANÇAMENTO no Senior X.")

    # Renomeando a coluna principal
    df_pendentes = df_pendentes.rename(columns={'Documento_Limpo': 'Número da NF não lançada'})

    # Organizando a ordem das colunas para o relatório ficar perfeito
    colunas_principais = ['Número da NF não lançada', 'Tipo_Doc', 'Razão Social Emissor', 'Total da NF-e', 'Data e Hora da Emissão']
    colunas_existentes = [col for col in colunas_principais if col in df_pendentes.columns]
    colunas_resto = [col for col in df_pendentes.columns if col not in colunas_existentes]

    df_relatorio_final = df_pendentes[colunas_existentes + colunas_resto]

    # SALVANDO NO EXCEL
    nome_saida = "Relatorio_NFs_e_CTes_Pendentes.xlsx"
    df_relatorio_final.to_excel(nome_saida, index=False)
    print(f"📋 Planilha completa '{nome_saida}' gerada com sucesso!")
    print("👉 Abra a pasta lateral esquerda (📁), atualize e baixe o arquivo para enviar no Outlook.")

    # Mostra todas as 37 linhas na tela para conferência imediata
    pd.set_option('display.max_rows', None)
    print("\n👀 CONFERÊNCIA RÁPIDA DOS DOCUMENTOS FALTANTES:")
    print(df_relatorio_final[colunas_existentes])
    pd.reset_option('display.max_rows')
else:
    print("✅ Espetacular! Todas as NF-es e CT-es ativas do eDocs constam lançadas no Senior X.")
print("========================================================")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

=== CONCILIAÇÃO FISCAL AUTOMATIZADA: NF-E + CTE ===
✅ Os três arquivos Excel foram carregados com sucesso!

================ RESULTADO DA AUDITORIA ================
⚠️ Sucesso! Foram encontrados 37 documentos (NF-e/CT-e) SEM LANÇAMENTO no Senior X.
📋 Planilha completa 'Relatorio_NFs_e_CTes_Pendentes.xlsx' gerada com sucesso!
👉 Abra a pasta lateral esquerda (📁), atualize e baixe o arquivo para enviar no Outlook.

👀 CONFERÊNCIA RÁPIDA DOS DOCUMENTOS FALTANTES:
     Número da NF não lançada Tipo_Doc  \
1                        9300     NF-e   
28                       9365     NF-e   
30                       9380     NF-e   
31                       9381     NF-e   
32                       9383     NF-e   
44                     134206     NF-e   
49                       9414     NF-e   
50                      61287     NF-e   
60                       9479     NF-e   
86                       9671     NF-e   
93                       5502     NF-e   
95                        778    